In [8]:
# pip install pymupdf
import fitz
from dataclasses import dataclass
from typing import List

@dataclass
class Section:
    level: int
    title: str
    start_page: int   # 0-based
    end_page: int     # inclusive, 0-based
    texts: List[str]
    text: str

def extract_top_level_sections(pdf_path: str) -> List[Section]:
    doc = fitz.open(pdf_path)
    try:
        # 1) 优先使用 TOC（目录）
        toc = doc.get_toc(simple=False)  # 1.24+ 返回扩展条目，兼容性更好 [web:23]
        entries = []
        if toc:
            if isinstance(toc[0], dict):
                # 规范化为 (lvl, title, pno0) 且 pno 1-based -> 0-based [web:23][web:1]
                for item in toc:
                    lvl = int(item.get("level", 1))
                    title = str(item.get("title", "")).strip()
                    pno0 = int(item.get("page", 1)) - 1
                    if title and pno0 >= 0:
                        entries.append((lvl, title, pno0))
            else:
                for it in toc:
                    if len(it) >= 3:
                        lvl, title, pno1 = it[0], it[1], it[2]
                        title = (title or "").strip()
                        pno0 = int(pno1) - 1
                        if title and pno0 >= 0:
                            entries.append((int(lvl), title, pno0))

        sections: List[Section] = []

        if entries:
            # 2) 只保留 level == 1 的顶级章节 [web:1][web:23]
            top = [(lvl, title, pno) for (lvl, title, pno) in entries if lvl == 1]
            top.sort(key=lambda x: x[2])
            # 计算每个顶级章节的结束页：下一顶级起始页-1，最后一个到文档末页 [web:1][web:23]
            for i, (_, title, start) in enumerate(top):
                end = doc.page_count - 1
                if i + 1 < len(top):
                    _, _, next_start = top[i + 1]
                    end = max(start, next_start)
                texts = []
                for pno in range(start, end + 1):
                    page = doc.load_page(pno)
                    texts.append(page.get_text("text"))  # 简洁模式提取正文 [web:31][web:34]
                sections.append(Section(level=1, title=title, start_page=start, end_page=end, text="\n".join(texts), texts=texts))
            return sections

        # 3) 无 TOC，启发式：用最大字号标题作为“第一章”锚点 [web:31][web:34]
        anchors = []  # (page, max_span_size, text)
        for pno in range(doc.page_count):
            page = doc.load_page(pno)
            data = page.get_text("dict", flags=fitz.TEXTFLAGS_TEXT)  # 取 spans 信息 [web:31]
            for blk in data.get("blocks", []):
                for line in blk.get("lines", []):
                    spans = line.get("spans", [])
                    if not spans:
                        continue
                    text = "".join(s.get("text", "") for s in spans).strip()
                    if not text:
                        continue
                    sizes = [s.get("size", 0.0) for s in spans]
                    max_size = max(sizes) if sizes else 0.0
                    if len(text) <= 160:  # 限制行长，更像标题 [web:31]
                        anchors.append((pno, max_size, text))

        if not anchors:
            # 整份文档作为单个顶级章节 [web:31]
            all_text = []
            for p in range(doc.page_count):
                all_text.append(doc.load_page(p).get_text("text"))
            return [Section(level=1, title="Document", start_page=1, end_page=doc.page_count, text="\n".join(all_text), texts=texts)]

        # 求全局字号阈值，筛选为“第一层标题”
        anchors.sort(key=lambda x: (-x[1], x[0]))
        top_size = anchors[0][1]
        thr = max(top_size * 0.92, top_size - 0.5)  # 稍放宽 [web:34]
        lvl1 = sorted([(p, s, t) for (p, s, t) in anchors if s >= thr], key=lambda x: x[0])

        # 合并相邻页重复标题
        merged = []
        last_title = None
        for p, s, t in lvl1:
            if last_title and t == last_title:
                continue
            merged.append((p, t))
            last_title = t

        # 切分范围并提取正文
        for i, (start, title) in enumerate(merged):
            end = doc.page_count - 1
            if i + 1 < len(merged):
                end = max(start, merged[i + 1][0])
            texts = []
            for pno in range(start, end + 1):
                texts.append(doc.load_page(pno).get_text("text"))
            sections.append(Section(level=1, title=title.strip(), start_page=start, end_page=end, text="\n".join(texts), texts=texts))
        return sections

    finally:
        doc.close()

        
path = "/home/snt/projects_lujun/mt_reasoning/data/source/luxembourgish_grammar.pdf"
sections = extract_top_level_sections(path)
for i, sec in enumerate(sections, 1):
    print(f"[{i}] {sec.title} | pages {sec.start_page}-{sec.end_page}")
    print(f"Chars: {len(sec.text)}")
    print(f"Pages_count: {len(sec.texts)}")
    print("-" * 80)


[1] Luxembourgish | pages 0-1
Chars: 3241
Pages_count: 2
--------------------------------------------------------------------------------
[2] 1. Introduction | pages 1-2
Chars: 3294
Pages_count: 2
--------------------------------------------------------------------------------
[3] 2. Sketch of the Sociohistorical and Sociolinguistic | pages 2-2
Chars: 1531
Pages_count: 1
--------------------------------------------------------------------------------
[4] Evolution | pages 2-8
Chars: 12227
Pages_count: 7
--------------------------------------------------------------------------------
[5] 3. Phonetics and Phonology | pages 8-20
Chars: 22915
Pages_count: 13
--------------------------------------------------------------------------------
[6] 4. Morphosyntax | pages 20-46
Chars: 45191
Pages_count: 27
--------------------------------------------------------------------------------
[7] 5. Selected Syntactic Characteristics | pages 46-47
Chars: 3177
Pages_count: 2
-----------------------------

In [9]:
import fitz  # PyMuPDF
import os

def pdf_to_images(pdf_path, out_dir, dpi=300, fmt="png"):
    os.makedirs(out_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    try:
        # 计算缩放：72dpi 基准，300/72 ≈ 4.1667
        zoom = dpi / 72.0
        mat = fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=mat, alpha=False)  # 关闭 alpha 有助于减小文件体积
            out_path = os.path.join(out_dir, f"page_{i:04d}.{fmt}")
            pix.save(out_path)
    finally:
        doc.close()



In [10]:
from PIL import Image
import os
from glob import glob
from typing import List, Tuple, Optional
from typing import List, Tuple, Optional, Union

def stitch_images_in_pairs(
    in_dir: str,
    out_dir: str,
    pattern: str = "page_*.png",
    bg_color: Optional[Tuple[int, int, int]] = (255, 255, 255),  # None 表示保留透明通道（若源图有）
    align: str = "center",  # "left" | "center" | "right"
    pad: int = 0,           # 上下左右内边距
    # 新增：裁剪控制
    footer_crop_first: Union[int, float] = 0,   # 第一张：去底部高度（像素或比例，如 120 或 0.06）
    header_crop_second: Union[int, float] = 0,  # 第二张：去顶部高度（像素或比例，如 110 或 0.05）
) -> List[str]:
    """
    将输入目录中的页面图片按文件名排序，每两张纵向拼接为一张长图。
    - 第一张去除页脚，第二张去除页眉。
    - footer_crop_first/header_crop_second 支持像素(int)或相对高度比例(float, 0~1)。
    - 若总数为奇数，最后一张单独输出（仅做第一张的页脚裁剪）。
    - 输出文件名为 pair_0000.png, pair_0001.png, ...
    """
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(glob(os.path.join(in_dir, pattern)))
    outputs = []

    def open_img(fp: str) -> Image.Image:
        img = Image.open(fp)
        img.load()
        return img

    def place_x_offset(canvas_w: int, img_w: int, align: str) -> int:
        if align == "left":
            return pad
        elif align == "right":
            return canvas_w - img_w - pad
        else:
            return (canvas_w - img_w) // 2

    def _resolve_crop(value: Union[int, float], total: int) -> int:
        if isinstance(value, float):
            px = int(round(total * value))
        else:
            px = int(value)
        return max(0, min(px, total))

    def crop_header_footer(img: Image.Image, top_crop: Union[int, float]=0, bottom_crop: Union[int, float]=0) -> Image.Image:
        h = img.height
        t = _resolve_crop(top_crop, h)
        b = _resolve_crop(bottom_crop, h)
        if t + b >= h:
            b = max(0, h - t - 1)
        return img.crop((0, t, img.width, h - b))

    i = 0
    pair_idx = 0
    n = len(files)

    while i < n:
        img1 = open_img(files[i])
        img2 = open_img(files[i+1]) if i+1 < n else None

        # 裁剪：第一张去底部、第二张去顶部
        img1_c = crop_header_footer(img1, top_crop=0, bottom_crop=footer_crop_first)
        if img2 is not None:
            img2_c = crop_header_footer(img2, top_crop=header_crop_second, bottom_crop=0)
        else:
            img2_c = None

        # 计算画布尺寸
        widths = [img1_c.width] + ([img2_c.width] if img2_c is not None else [])
        heights = [img1_c.height] + ([img2_c.height] if img2_c is not None else [])
        canvas_w = max(widths) + pad * 2
        canvas_h = sum(heights) + pad * (3 if img2_c is not None else 2)

        # 背景/模式
        use_transparent = (bg_color is None)
        mode = "RGBA" if use_transparent else "RGB"
        bg = (0, 0, 0, 0) if use_transparent else bg_color
        canvas = Image.new(mode, (canvas_w, canvas_h), bg)

        # 粘贴第一张（已裁剪）
        x1 = place_x_offset(canvas_w, img1_c.width, align)
        y1 = pad
        if use_transparent and img1_c.mode in ("RGBA", "LA"):
            canvas.paste(img1_c, (x1, y1), img1_c)
        else:
            canvas.paste(img1_c, (x1, y1))

        # 粘贴第二张（已裁剪）
        if img2_c is not None:
            x2 = place_x_offset(canvas_w, img2_c.width, align)
            y2 = y1 + img1_c.height + pad
            if use_transparent and img2_c.mode in ("RGBA", "LA"):
                canvas.paste(img2_c, (x2, y2), img2_c)
            else:
                canvas.paste(img2_c, (x2, y2))

        out_path = os.path.join(out_dir, f"pair_{pair_idx:04d}.png")
        canvas.save(out_path)
        outputs.append(out_path)

        pair_idx += 1
        i += 1  # 成对前进

        # 释放资源
        img1.close()
        if img2 is not None:
            img2.close()

    return outputs



In [11]:
import os
import cv2
import numpy as np
from typing import List
from paddleocr import TableRecognitionPipelineV2

_PIPELINE_CACHE = {}

def _get_table_pipeline(use_gpu: bool):
    key = ("gpu" if use_gpu else "cpu")
    if key not in _PIPELINE_CACHE:
        device = "gpu:0" if use_gpu else "cpu"
        _PIPELINE_CACHE[key] = TableRecognitionPipelineV2(
            text_recognition_model_name="latin_PP-OCRv5_mobile_rec",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            device=device
        )
    return _PIPELINE_CACHE[key]

def save_table_crops(img_path: str, out_dir: str = "crops", use_gpu: bool = False) -> List[str]:
    """
    Detect tables with TableRecognitionPipelineV2 and save each table crop into out_dir.
    Returns a list of saved file paths.
    """
    os.makedirs(out_dir, exist_ok=True)
    device = "gpu:0" if use_gpu else "cpu"

    pipeline = _get_table_pipeline(use_gpu)  # 复用已有的
    padding = 10

    results = pipeline.predict(img_path)
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Failed to read image: {img_path}")
    h, w = img.shape[:2]
    base = os.path.splitext(os.path.basename(img_path))[0]

    saved_paths: List[str] = []
    tbl_idx = 1

    for res in results:
        data = res.json  # same structure as save_to_json
        # 1) Prefer union of cell boxes for tight table crop
        table_list = data.get("res", {}).get("table_res_list", [])
        if table_list:
            for tbl in table_list:
                cell_polys = tbl.get("cell_box_list", [])
                if len(cell_polys) > 0:
                    pts = []
                    for poly in cell_polys:
                        p = np.asarray(poly).reshape(-1, 2)
                        pts.append(p)
                    pts = np.vstack(pts)
                    x_min = max(int(np.floor(pts[:, 0].min())) - padding, 0)
                    y_min = max(int(np.floor(pts[:, 1].min())) - padding, 0)
                    x_max = min(int(np.ceil(pts[:, 0].max())) + padding, w - 1)
                    y_max = min(int(np.ceil(pts[:, 1].max())) + padding, h - 1)
                    if x_max > x_min and y_max > y_min:
                        crop = img[y_min:y_max, x_min:x_max]
                        save_path = os.path.join(out_dir, f"{base}_table{tbl_idx}.png")
                        cv2.imwrite(save_path, crop)
                        saved_paths.append(save_path)
                        tbl_idx += 1

        # 2) Fallback: use layout table boxes if no cell boxes or no tables above
        if len(saved_paths) == 0:
            layout_boxes = data.get("res", {}).get("layout_det_res", {}).get("boxes", [])
            for b in layout_boxes:
                if b.get("label", "") == "table":
                    x1, y1, x2, y2 = b["coordinate"]
                    x1 = max(int(np.floor(x1)) - padding, 0)
                    y1 = max(int(np.floor(y1)) - padding, 0)
                    x2 = min(int(np.ceil(x2)) + padding, w - 1)
                    y2 = min(int(np.ceil(y2)) + padding, h - 1)
                    if x2 > x1 and y2 > y1:
                        crop = img[y1:y2, x1:x2]
                        save_path = os.path.join(out_dir, f"{base}_table{tbl_idx}.png")
                        cv2.imwrite(save_path, crop)
                        saved_paths.append(save_path)
                        tbl_idx += 1

    return saved_paths




In [12]:
import os
from paddleocr import TableRecognitionPipelineV2
import pandas as pd
_PIPELINE_CACHE = {}

def _get_table_pipeline(use_gpu: bool):
    key = ("gpu" if use_gpu else "cpu")
    if key not in _PIPELINE_CACHE:
        device = "gpu:0" if use_gpu else "cpu"
        _PIPELINE_CACHE[key] = TableRecognitionPipelineV2(
            layout_detection_model_name= "PP-DocLayout-L",
            table_classification_model_name="PP-LCNet_x1_0_table_cls",
            text_recognition_model_name="latin_PP-OCRv5_mobile_rec",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            device=device,
            use_layout_detection = True,
        )
    return _PIPELINE_CACHE[key]

def run_table_ocr(base_dir, img_name, out_dir, use_gpu=False):
    
    os.makedirs(out_dir, exist_ok=True)
    device = "gpu:0" if use_gpu else "cpu"
    pipeline = _get_table_pipeline(use_gpu)
    img_path = os.path.join(base_dir, img_name)
    results = pipeline.predict(img_path, use_layout_detection = True,)

    for idx, res in enumerate(results):
        res.save_to_img(out_dir)
        res.save_to_html(out_dir)
        res.save_to_xlsx(out_dir)
        res.save_to_json(out_dir)

    tables_dict_list = []
    for html_file_name in os.listdir(out_dir):
        print (html_file_name)
        if html_file_name.endswith(".html"):
            if img_name.replace(".png", "") in html_file_name:
                html_path = os.path.join(out_dir, html_file_name)
                print(f"HTML output found in: {html_path}")
                tables = pd.read_html(html_path)  # 返回列表
                if not tables:
                    print("No tables found in the HTML file.")
                    continue
                df = tables[0]  # 假设只取第一个表格

                # 2. 删除空行和空列
                df = df.dropna(how="all")      # 删除全空的行
                df = df.dropna(axis=1, how="all")  # 删除全空的列

                html_code = df.to_html(index=False, escape=False)

                json_code = df.to_json(orient="records", force_ascii=False, indent=4)

                markdown_code = df.to_markdown(index=False)

                tables_dict_list.append({
                    "df": df,
                    "html": html_code,
                    "json": json_code,
                    "markdown": markdown_code
                })
    return tables_dict_list

# tables_dict_list = run_table_ocr(base_dir="data/extraction_pdf/pages", img_name="page_0029.png", out_dir="data/extraction_pdf/after_crops_tables", use_gpu=True)

## Put All Together

In [13]:

## Get Sections, Pages numbers
path = "/home/snt/projects_lujun/mt_reasoning/data/source/luxembourgish_grammar.pdf"
sections = extract_top_level_sections(path)
# for i, sec in enumerate(sections, 1):
#     print(f"[{i}] {sec.title} | pages {sec.start_page}-{sec.end_page}")
#     print(f"Chars: {len(sec.text)}")
#     print("-" * 80)

## Save all images of pages
# imgs = pdf_to_images("/home/snt/projects_lujun/mt_reasoning/data/source/luxembourgish_grammar.pdf", "data/extraction_pdf/pages", dpi=300, fmt="png")

# pairs = stitch_images_in_pairs(
#     in_dir="data/extraction_pdf/pages",   
#     out_dir="data/extraction_pdf/pages_pairs", 
#     pattern="page_*.png",
#     bg_color=(255, 255, 255),             # 白底；如需透明底改为 None
#     align="center",                       # 宽度不一时水平居中
#     pad=20 ,                               # 四周留 20px 内边距
#     footer_crop_first=0.08,   # 第一张去掉底部 6%
#     header_crop_second=0.08,  # 第二张去掉顶部 6%
# )


## Save table crops from all page images
# for file in os.listdir("data/extraction_pdf/pages"):
#     if file.endswith(".png"):
#         img_path = os.path.join("data/extraction_pdf/pages", file)
#         save_table_crops(img_path, "data/extraction_pdf/crops", use_gpu=True)


# `extraction_list.pkl` Data Structure

The file `extraction_list.pkl` is a pickled Python object containing the structured extraction results from a PDF.  

## Top-Level Structure

```python
extraction_list = [
    {
        "section_title": str,            # Title of the section/chapter
        "start_page": int,               # First page number of the section
        "end_page": int,                 # Last page number of the section
        "content_dict_list": [           # Ordered list of content elements
            {
                "dtype": "text",         # Content type ("text" or "table")
                "text": str,             # (for text) The extracted paragraph text
                "page_num": int          # Page number where this text appears
            },
            {
                "dtype": "table",        # Table entry
                "df": pandas.DataFrame,  # (for table) The table as a DataFrame
                "html": str,             # Table in HTML format
                "json": dict,            # Table in JSON format
                "markdown": str,         # Table in Markdown format
                "page_num": int          # Page number where this table appears
            },
            ...
        ]
    },
    ...
]


In [14]:
extraction_list = []
for i, sec in enumerate(sections, 1):
    # Each Chapiter
    chapiter_dict = {}
    chapiter_dict["section_title"] = sec.title
    chapiter_dict["start_page"] = sec.start_page
    chapiter_dict["end_page"] = sec.end_page

    chapiter_dict["content_dict_list"] = []
    for i, text in enumerate(sec.texts, 1):
        content_dict = {}
        content_dict["dtype"] = "text"
        content_dict["text"] = text
        content_dict["page_num"] = sec.start_page + i - 1
        chapiter_dict["content_dict_list"].append(content_dict)

    extraction_list.append(chapiter_dict)
        

for file_name in os.listdir("data/extraction_pdf/pages"):
    if not file_name.endswith(".png"):
        continue

    table_list = run_table_ocr("data/extraction_pdf/pages", file_name, "data/extraction_pdf/after_crops_tables", use_gpu=True)

    if table_list is None or len(table_list) == 0:
        continue
    
    for table_dict in table_list:
        # insert the table one by one
        page_num = int(file_name.replace("page_", "").replace(".png", ""))
        # find the chpiter that contains this page number
        for extraction in extraction_list:
            if page_num >= extraction["start_page"] and page_num <= extraction["end_page"]:
                for i, content_dict in enumerate(extraction["content_dict_list"]):
                    start_page = extraction["start_page"]
                    end_page = extraction["end_page"]
                    if page_num >= start_page+i and page_num <= end_page:
                        index_insert = i
                        for content_dict in extraction["content_dict_list"]:
                            if content_dict["dtype"] == "text" and content_dict["page_num"] == page_num:
                                index_insert = extraction["content_dict_list"].index(content_dict) + 1
                                break
                        dict_to_insert = {}
                        dict_to_insert["dtype"] = "table"
                        dict_to_insert["df"] = table_dict["df"]
                        dict_to_insert["html"] = table_dict["html"]
                        dict_to_insert["json"] = table_dict["json"]
                        dict_to_insert["markdown"] = table_dict["markdown"]
                        dict_to_insert["page_num"] = page_num
                        extraction["content_dict_list"].insert(index_insert, dict_to_insert)
                        print(f"Insert table into section {extraction['section_title']} at page {page_num} position {index_insert}")
                        break
    
import pickle

with open("extraction_list.pkl", "wb") as f:
    pickle.dump(extraction_list, f)



Creating model: ('PP-DocLayout-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/snt/.paddlex/official_models/PP-DocLayout-L`.
Creating model: ('PP-LCNet_x1_0_table_cls', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/snt/.paddlex/official_models/PP-LCNet_x1_0_table_cls`.
Creating model: ('SLANeXt_wired', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/snt/.paddlex/official_models/SLANeXt_wired`.
Creating model: ('SLANeXt_wireless', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/snt/.paddlex/official_models/SLANeXt_wireless`.
Creating model: ('RT-DETR-L_wired_table_cell_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/snt/.paddlex/official_models/RT-DETR-L_w

page_0006_ocr_res_img.png
page_0027_preprocessed_img.png
page_0029_ocr_res_img.png
page_0006_preprocessed_img.png
page_0011_table_2.html
page_0007_res.json
page_0007_layout_det_res.png
page_0027_layout_det_res.png
page_0011_table_1.html
page_0029_table_1.xlsx
page_0017_table_1.html
page_0017_res.json
page_0006_res.json
page_0044_table_2.html
page_0011_ocr_res_img.png
page_0029_preprocessed_img.png
page_0011_res.json
page_0017_table_1.xlsx
page_0006_layout_det_res.png
page_0029_table_1.html
page_0044_table_2.xlsx
page_0000_res.json
page_0027_res.json
page_0044_preprocessed_img.png
page_0044_ocr_res_img.png
page_0000_ocr_res_img.png
page_0017_ocr_res_img.png
page_0044_res.json
page_0044_layout_det_res.png
page_0044_table_1.html
page_0017_layout_det_res.png
page_0000_layout_det_res.png
page_0011_layout_det_res.png
page_0011_table_1.xlsx
page_0044_table_cell_img.png
page_0017_preprocessed_img.png
page_0044_table_1.xlsx
page_0027_table_1.html
page_0027_table_cell_img.png
page_0011_table_cel

In [ ]:
with open("extraction_list.pkl", "rb") as f:
    extraction_list_new = pickle.load(f)

count_table = 0
for extraction in extraction_list_new:
    content_dict_list = extraction["content_dict_list"]
    for content_dict in content_dict_list:
        if content_dict["dtype"] == "table":
            count_table += 1
print(f"Total tables extracted: {count_table}")

Total tables extracted: 34
